In [1]:
# Proyecto Final: Carga de datos:
!mkdir -p DATOS
!wget "https://github.com/josepgarcia/datos/raw/refs/heads/main/stocks.txt" -P ./DATOS
!head ./DATOS/stocks.txt

--2026-11-19 00:41:40--  https://github.com/josepgarcia/datos/raw/refs/heads/main/stocks.txt
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/josepgarcia/datos/refs/heads/main/stocks.txt [following]
--2026-11-19 00:41:40--  https://raw.githubusercontent.com/josepgarcia/datos/refs/heads/main/stocks.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 986 [text/plain]
Saving to: ‘./DATOS/stocks.txt’

stocks.txt          100%[===================>]     986  --.-KB/s    in 0s      

2026-11-19 00:41:41 (12.2 MB/s) - ‘./DATOS/stocks.txt’ saved [986/986]

id,name,category,quantity,price
1,iPhone,Electron

In [2]:
# P1: Config. Inicial: 
from pyspark.sql import SparkSession
import os
spark = SparkSession.builder \
    .appName("AnalisisStocks") \
    .config("spark.executor.memory", "512m") \
    .getOrCreate()
sc = spark.sparkContext
path_relativo = "./DATOS/stocks.txt"
path_absoluto = os.path.abspath(path_relativo)
path_final = "file://" + path_absoluto
rdd = sc.textFile(path_final)
print(rdd.take(5))

['id,name,category,quantity,price', '1,iPhone,Electronics,10,899.99', '2,Macbook,Electronics,5,1299.99', '3,iPad,Electronics,15,499.99', '4,Samsung TV,Electronics,8,799.99']


In [3]:
# P2: Limpieza y Parseo de Datos:
def parsear_linea(linea):
    try:
        partes = linea.split(",")
        # Formato REAL: id,name,category,quantity,price
        simbolo = partes[2]            # category  -> "símbolo"
        fecha   = partes[1]            # name      -> "fecha"
        precio  = float(partes[4])     # price
        volumen = int(partes[3])       # quantity  -> "volumen"
        return (simbolo, fecha, precio, volumen)
    except:
        return None                    # el encabezado cae aquí y se descarta

rdd_parseado = rdd.map(parsear_linea).filter(lambda x: x is not None)
print(f"Líneas válidas: {rdd_parseado.count()}")   # 30

Líneas válidas: 30


In [4]:
# P3: Estadisticas por simbolo:
simbolo_precio = rdd_parseado.map(lambda x: (x[0], x[2]))
precio_max = simbolo_precio.reduceByKey(lambda a, b: max(a, b))
precio_min = simbolo_precio.reduceByKey(lambda a, b: min(a, b))
precio_promedio = simbolo_precio.aggregateByKey(
    (0.0, 0),
    lambda acc, valor: (acc[0] + valor, acc[1] + 1),
    lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1])
).mapValues(lambda x: x[0] / x[1])

print("Precio máximo por símbolo:")
for simbolo, precio in sorted(precio_max.collect()):
    print(f"{simbolo}: ${precio:.2f}")
# Repite el for con precio_min y precio_promedio

Precio máximo por símbolo:
Accessories: $29.99
Clothing: $199.99
Electronics: $1299.99
Food: $5.99
Furniture: $249.99
Sports: $49.99


In [5]:
# P4: Top 10 Dias mayor actividad.

fecha_volumen = rdd_parseado.map(lambda x: (x[1], x[3]))
volumen_por_fecha = fecha_volumen.reduceByKey(lambda a, b: a + b)
top_fechas = volumen_por_fecha.sortBy(lambda x: x[1], ascending=False)
print("Top 10 con mayor volumen:")
for fecha, volumen in top_fechas.take(10):
    print(f"{fecha}: {volumen:,} unidades")

Top 10 con mayor volumen:
Bananas: 150 unidades
Oranges: 120 unidades
Apples: 100 unidades
Chicken Breast: 50 unidades
T-shirt: 50 unidades
Sneakers: 40 unidades
Backpack: 30 unidades
Nike Shoes: 30 unidades
Salmon Fillet: 30 unidades
Jeans: 30 unidades


In [7]:
# P5: Volatilidad Por Simbolo:
stats_por_simbolo = simbolo_precio.aggregateByKey(
    (float('-inf'), float('inf')),
    lambda acc, precio: (max(acc[0], precio), min(acc[1], precio)),
    lambda acc1, acc2: (max(acc1[0], acc2[0]), min(acc1[1], acc2[1]))
)
volatilidad = stats_por_simbolo.mapValues(lambda x: ((x[0] - x[1]) / x[1]) * 100)
mas_volatiles = volatilidad.sortBy(lambda x: x[1], ascending=False).take(5)
mas_estables  = volatilidad.sortBy(lambda x: x[1], ascending=True).take(5)
print("Más volátiles:");  [print(f"{s}: {v:.2f}%") for s, v in mas_volatiles]
print("Más estables:");   [print(f"{s}: {v:.2f}%") for s, v in mas_estables]

Más volátiles:
Food: 2296.00%
Clothing: 1234.16%
Electronics: 900.07%
Furniture: 212.53%
Sports: 150.08%
Más estables:
Accessories: 20.01%
Sports: 150.08%
Furniture: 212.53%
Electronics: 900.07%
Clothing: 1234.16%


[None, None, None, None, None]

In [8]:
# P6: Guardar.
!mkdir -p RESULTADOS

In [9]:
# TODO: Guardar resultados 
output_path_volumen = "file://" + os.path.abspath("./RESULTADOS/volumen_por_fecha")
volumen_por_fecha.saveAsTextFile(output_path_volumen)

output_path_stats = "file://" + os.path.abspath("./RESULTADOS/stats_por_simbolo")
stats_por_simbolo.saveAsTextFile(output_path_stats)

# TODO: Calcular métricas para resumen
total_registros = rdd_parseado.count()
simbolos_unicos = rdd_parseado.map(lambda x: x[0]).distinct().count()
simbolo_mayor_precio = precio_promedio.sortBy(lambda x: x[1], ascending=False).first()
dia_mayor_volumen = volumen_por_fecha.sortBy(lambda x: x[1], ascending=False).first()
simbolo_mas_volatil = volatilidad.sortBy(lambda x: x[1], ascending=False).first()

# TODO: Mostrar resumen ejecutivo
print("\n" + "="*60)
print("RESUMEN EJECUTIVO - ANÁLISIS DE STOCKS")
print("="*60)
print(f"\nTotal de registros procesados: {total_registros:,}")
print(f"Número de símbolos únicos: {simbolos_unicos}")
print(f"\nSímbolo con mayor precio promedio: {simbolo_mayor_precio[0]} (${simbolo_mayor_precio[1]:.2f})")
print(f"Día con mayor volumen: {dia_mayor_volumen[0]} ({dia_mayor_volumen[1]:,} acciones)")
print(f"Símbolo más volátil: {simbolo_mas_volatil[0]} ({simbolo_mas_volatil[1]:.2f}%)")

# TODO: Cerrar sesión
spark.stop()
print("\nAnálisis completado. Sesión cerrada.")


RESUMEN EJECUTIVO - ANÁLISIS DE STOCKS

Total de registros procesados: 30
Número de símbolos únicos: 6

Símbolo con mayor precio promedio: Electronics ($586.66)
Día con mayor volumen: Bananas (150 acciones)
Símbolo más volátil: Food (2296.00%)

✅ Análisis completado. Sesión cerrada.
